In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing   import LabelEncoder
from sklearn.linear_model    import LogisticRegression

import lightgbm as lgb
import xgboost  as xgb
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [4]:
PATH  = "/kaggle/input/datasets/ramjasmaurya/full-zindi-dataset/"

train = pd.read_csv(PATH + 'Train(1).csv')
test  = pd.read_csv(PATH + 'Test.csv')
# sub   = pd.read_csv(PATH + 'sample_submission.csv')

print(f'Train : {train.shape} | Test : {test.shape}')

Train : (2154021, 13) | Test : (280961, 13)


In [5]:
unique_locations = train[["lat", "lon"]].drop_duplicates()

print(unique_locations.shape)
print(unique_locations.head())

(15715, 2)
    lat   lon
0 -55.5 -68.5
1 -55.5 -67.5
2 -54.5 -71.5
3 -54.5 -70.5
4 -54.5 -69.5


                     TRAIN
    ------------------------------------------------
    
    SPEI
    SOIL
    Location
    Month
          │
          ▼
     Model A
          │
          ▼
    Predict TWS_t
    
    
    TWS_t
    SPEI
    SOIL
    Month
    Location
          │
          ▼
     Model B
          │
          ▼
    Predict TWS_(t+1)
    
    
    
                     TEST
    ------------------------------------------------
    
    Missing TWS_t
          │
          ▼
     Model A
          │
    Filled TWS_t
          │
          ▼
     Model B
          │
          ▼
    Final submission



Yes, and I think it's actually a good direction, with one caveat.

For Model 1 (reconstructing TWS_t), your goal is to learn the physical relationship between drought, soil moisture, seasonality, and TWS—not to memorize specific locations or years.

So I'd think of the features in three categories:

Feature	Use?	Reason
SPEI_01_t, SPEI_03_t, SPEI_06_t, SPEI_12_t	✅ Yes	Direct hydrological drivers
SOIL_MOISTURE_t	✅ Yes	Strong physical relationship with TWS
month_sin, month_cos	✅ Yes	Represents seasonality without encoding specific years
lat, lon	⚠️ Maybe	Useful if climate differs strongly by location, but can encourage memorization
time	❌ No	Future dates won't have been seen during training
sample_id	❌ No	Pure identifier
target	❌ No	That's the forecasting target, never an input
My preferred progression

Start with the most general model:

SPEI_01_t
SPEI_03_t
SPEI_06_t
SPEI_12_t
SOIL_MOISTURE_t
month_sin
month_cos
        ↓
      TWS_t

This model learns a mapping driven by hydrology and seasonality rather than by specific coordinates.

If its RMSE is good enough, stop there.

If not, then gradually add spatial information:

+ lat
+ lon

or even better,

+ elevation
+ climate zone
+ basin ID

if those are available, since they are more physically meaningful than raw coordinates.

One thing I would avoid

I would not use features like:

year
time
time_spatial_group
lag features (TWS_lag1, etc.)

for Model 1 if the objective is to reconstruct missing TWS_t in unseen future months. Those features either won't exist at inference time or encourage the model to exploit temporal patterns that don't generalize as well.

My recommendation

I'd build two reconstruction models and compare them:

Model A (physics-based):

SPEI + SOIL + month_sin + month_cos

Model B (physics + spatial):

SPEI + SOIL + month_sin + month_cos + lat + lon

Evaluate both with the same temporal cross-validation. If Model B gives only a small improvement, I'd choose Model A because it's likely to generalize better to truly unseen months and years. If Model B provides a substantial gain without degrading temporal validation performance, then include the spatial features. This comparison will tell you whether location adds genuine predictive information or mostly helps the model memorize the training data.

In [ ]:
# def model_1_feature_engineer(df):

#     df = df.copy()

#     spei_t = [
#         "SPEI_01_t",
#         "SPEI_03_t",
#         "SPEI_06_t",
#         "SPEI_12_t",
#     ]

#     # 1. SPEI relationships (excellent)
    
#     df["SPEI_median"] = df[spei_t].median(axis=1)

#     df["SPEI_abs_mean"] = df[spei_t].abs().mean(axis=1)
    
#     df["SPEI_abs_max"] = df[spei_t].abs().max(axis=1)
    
#     df["SPEI_energy"] = (
#         df["SPEI_01_t"]**2 +
#         df["SPEI_03_t"]**2 +
#         df["SPEI_06_t"]**2 +
#         df["SPEI_12_t"]**2
#     )
    
#     df["SPEI_cv"] = (
#         df["SPEI_std_t"] /
#         (np.abs(df["SPEI_mean_t"]) + 1e-6)
#     )

#     # 2. Drought consistency

#     df["spei_sign_changes"] = (
#         np.sign(df["SPEI_01_t"]) != np.sign(df["SPEI_03_t"])
#     ).astype(int)
    
#     df["spei_all_same_sign"] = (
#         (
#             np.sign(df[spei_t])
#             .nunique(axis=1)
#             == 1
#         ).astype(int)
#     )

#     # 3. Multi-scale ratios

#     eps = 1e-6
    
#     df["SPEI01_div03"] = df["SPEI_01_t"] / (np.abs(df["SPEI_03_t"])+eps)
    
#     df["SPEI03_div06"] = df["SPEI_03_t"] / (np.abs(df["SPEI_06_t"])+eps)
    
#     df["SPEI06_div12"] = df["SPEI_06_t"] / (np.abs(df["SPEI_12_t"])+eps)

#     # 4. Soil transformations

#     df["soil_abs"] = np.abs(df["SOIL_MOISTURE_t"])
    
#     df["soil_cube"] = df["SOIL_MOISTURE_t"]**3
    
#     df["soil_inv"] = 1/(np.abs(df["SOIL_MOISTURE_t"])+1)

#     # 5. Soil × SPEI interactions
    
#     for c in spei_t:
#         df[f"soil_x_{c}"] = (
#             df["SOIL_MOISTURE_t"] *
#             df[c]
#         )
    
#         df[f"soil_div_{c}"] = (
#             df["SOIL_MOISTURE_t"] /
#             (np.abs(df[c])+1e-6)
#         )

#     # 6. Wetness index
    
#     df["wetness_index"] = (
#         df["SOIL_MOISTURE_t"] *
#         df["SPEI_mean_t"]
#     )
    
#     df["wetness_abs"] = (
#         np.abs(df["SOIL_MOISTURE_t"]) *
#         np.abs(df["SPEI_mean_t"])
#     )

#     # 7. Drought severity classes
    
#     df["severe_drought"] = (
#         df["SPEI_mean_t"] < -2
#     ).astype(np.int8)
    
#     df["moderate_drought"] = (
#         df["SPEI_mean_t"] < -1
#     ).astype(np.int8)
    
#     df["wet_condition"] = (
#         df["SPEI_mean_t"] > 1
#     ).astype(np.int8)

#     # 8. Nonlinear SPEI

#     for c in spei_t:
    
#         df[f"{c}_sq"] = df[c]**2
    
#         df[f"{c}_cube"] = df[c]**3
    
#         df[f"{c}_abs"] = np.abs(df[c])

#     # 9. Rank among SPEIs
    
    
#     df["SPEI_max_scale"] = np.argmax(
#         df[spei_t].values,
#         axis=1
#     )
    
#     df["SPEI_min_scale"] = np.argmin(
#         df[spei_t].values,
#         axis=1
#     )

#     # 10. Entropy-like variability
    
#     tmp = np.abs(df[spei_t])
    
#     p = tmp.div(tmp.sum(axis=1)+1e-6, axis=0)
    
#     df["SPEI_entropy"] = (
#         -(p*np.log(p+1e-6)).sum(axis=1)
#     )

#     return df

In [6]:
import numpy as np
import pandas as pd

def model_1_feature_engineer(df):

    df = df.copy()

    spei_t = [
        "SPEI_01_t",
        "SPEI_03_t",
        "SPEI_06_t",
        "SPEI_12_t",
    ]

    # --------------------------------------------------------
    # 0. Base Aggregations (Must be defined first)
    # --------------------------------------------------------
    df["SPEI_mean_t"] = df[spei_t].mean(axis=1)
    df["SPEI_std_t"] = df[spei_t].std(axis=1)

    # --------------------------------------------------------
    # 1. SPEI relationships (excellent)
    # --------------------------------------------------------
    
    df["SPEI_median"] = df[spei_t].median(axis=1)
    df["SPEI_abs_mean"] = df[spei_t].abs().mean(axis=1)
    df["SPEI_abs_max"] = df[spei_t].abs().max(axis=1)
    
    df["SPEI_energy"] = (
        df["SPEI_01_t"]**2 +
        df["SPEI_03_t"]**2 +
        df["SPEI_06_t"]**2 +
        df["SPEI_12_t"]**2
    )
    
    df["SPEI_cv"] = (
        df["SPEI_std_t"] /
        (np.abs(df["SPEI_mean_t"]) + 1e-6)
    )

    # --------------------------------------------------------
    # 2. Drought consistency
    # --------------------------------------------------------

    # Count sign changes across all adjacent scales (values 0 to 3)
    signs = np.sign(df[spei_t].values)
    df["spei_sign_changes"] = (
        (signs[:, 0] != signs[:, 1]).astype(int) +
        (signs[:, 1] != signs[:, 2]).astype(int) +
        (signs[:, 2] != signs[:, 3]).astype(int)
    )
    
    df["spei_all_same_sign"] = (
        (
            np.sign(df[spei_t])
            .nunique(axis=1)
            == 1
        ).astype(int)
    )

    # --------------------------------------------------------
    # 3. Multi-scale ratios
    # --------------------------------------------------------

    eps = 1e-6
    
    df["SPEI01_div03"] = df["SPEI_01_t"] / (np.abs(df["SPEI_03_t"]) + eps)
    df["SPEI03_div06"] = df["SPEI_03_t"] / (np.abs(df["SPEI_06_t"]) + eps)
    df["SPEI06_div12"] = df["SPEI_06_t"] / (np.abs(df["SPEI_12_t"]) + eps)

    # --------------------------------------------------------
    # 4. Soil transformations
    # --------------------------------------------------------

    df["soil_abs"] = np.abs(df["SOIL_MOISTURE_t"])
    df["soil_cube"] = df["SOIL_MOISTURE_t"]**3
    df["soil_inv"] = 1 / (np.abs(df["SOIL_MOISTURE_t"]) + 1)

    # --------------------------------------------------------
    # 5. Soil × SPEI interactions
    # --------------------------------------------------------
    
    for c in spei_t:
        df[f"soil_x_{c}"] = (
            df["SOIL_MOISTURE_t"] *
            df[c]
        )
    
        df[f"soil_div_{c}"] = (
            df["SOIL_MOISTURE_t"] /
            (np.abs(df[c]) + 1e-6)
        )

    # --------------------------------------------------------
    # 6. Wetness index
    # --------------------------------------------------------
    
    df["wetness_index"] = (
        df["SOIL_MOISTURE_t"] *
        df["SPEI_mean_t"]
    )
    
    df["wetness_abs"] = (
        np.abs(df["SOIL_MOISTURE_t"]) *
        np.abs(df["SPEI_mean_t"])
    )

    # --------------------------------------------------------
    # 7. Drought severity classes
    # --------------------------------------------------------
    
    df["severe_drought"] = (
        df["SPEI_mean_t"] < -2
    ).astype(np.int8)
    
    df["moderate_drought"] = (
        df["SPEI_mean_t"] < -1
    ).astype(np.int8)
    
    df["wet_condition"] = (
        df["SPEI_mean_t"] > 1
    ).astype(np.int8)

    # --------------------------------------------------------
    # 8. Nonlinear SPEI
    # --------------------------------------------------------

    for c in spei_t:
        df[f"{c}_sq"] = df[c]**2
        df[f"{c}_cube"] = df[c]**3
        df[f"{c}_abs"] = np.abs(df[c])

    # --------------------------------------------------------
    # 9. Rank among SPEIs (Optimized to np.int8)
    # --------------------------------------------------------
    
    df["SPEI_max_scale"] = (
        df[spei_t].values.argmax(axis=1).astype(np.int8)
    )
    
    df["SPEI_min_scale"] = (
        df[spei_t].values.argmin(axis=1).astype(np.int8)
    )

    # --------------------------------------------------------
    # 10. Entropy-like variability
    # --------------------------------------------------------
    
    tmp = np.abs(df[spei_t])
    p = tmp.div(tmp.sum(axis=1) + 1e-6, axis=0)
    
    df["SPEI_entropy"] = (
        -(p * np.log(p + 1e-6)).sum(axis=1)
    )

    return df

In [7]:
train = model_1_feature_engineer(train)
test = model_1_feature_engineer(test)

In [8]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

# -------------------------------------------------------
# RMSE
# -------------------------------------------------------
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))



IMPUTE_DROP = [
    "sample_id",
    "ID","lat","lon",
    "month_sin","month_cos", 	
    "Target",
    "target",
    "time",
    "TWS_t",
]

FEATURES = [
    c for c in train.columns
    if c not in IMPUTE_DROP
    and c in test.columns
]

# -------------------------------------------------------
# Train only where TWS_t exists
# -------------------------------------------------------
train_stage1 = train[train["TWS_t"].notna()].copy()

X = train_stage1[FEATURES]
y = train_stage1["TWS_t"]
X_test = test[FEATURES]

In [9]:
# -------------------------------------------------------
# CV
# -------------------------------------------------------
kf = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42,
)

oof_pred = np.zeros(len(train_stage1), dtype=np.float32)
test_pred = np.zeros(len(test), dtype=np.float32)

scores = []

# -------------------------------------------------------
# Training
# -------------------------------------------------------
for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):

    print("=" * 70)
    print(f"Fold {fold}")

    X_tr = X.iloc[tr_idx]
    y_tr = y.iloc[tr_idx]

    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]

    model = lgb.LGBMRegressor(
        objective="regression",
        metric="rmse",
        learning_rate=0.05,
        n_estimators=5000,
        num_leaves=127,
        subsample=0.8,
        colsample_bytree=0.75,
        random_state=42,
        n_jobs=-1,
    )

    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="rmse",
        callbacks=[
            lgb.early_stopping(200, verbose=False),
        ],
    )

    pred = model.predict(X_val)

    oof_pred[val_idx] = pred

    fold_rmse = rmse(y_val, pred)
    scores.append(fold_rmse)

    print(f"RMSE : {fold_rmse:.6f}")

    test_pred += model.predict(X_test) / kf.n_splits

# -------------------------------------------------------
# Results
# -------------------------------------------------------
print("=" * 70)
print("Fold RMSEs:")
print(np.round(scores, 6))

print(f"\nMean RMSE : {np.mean(scores):.6f}")
print(f"Std RMSE  : {np.std(scores):.6f}")

overall_rmse = rmse(y, oof_pred)
print(f"\nOOF RMSE  : {overall_rmse:.6f}")

# -------------------------------------------------------
# Save predictions
# -------------------------------------------------------
train_stage1["oof_pred"] = oof_pred
test["test_pred"] = test_pred

Fold 1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.652077 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10475
[LightGBM] [Info] Number of data points in the train set: 1938618, number of used features: 48
[LightGBM] [Info] Start training from score 0.117192
RMSE : 0.808635
Fold 2
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.683550 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10475
[LightGBM] [Info] Number of data points in the train set: 1938619, number of used features: 48
[LightGBM] [Info] Start training from score 0.117400
RMSE : 0.808318
Fold 3
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.650944 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10475
[LightGBM] [Info] Number of data points in the train set: 19

In [12]:
from sklearn.metrics import mean_squared_error
import numpy as np

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Rows where the true TWS_t is available
mask = test["TWS_t"].notna()

score = rmse(
    test.loc[mask, "TWS_t"],
    test.loc[mask, "test_pred"]
)

print(f"RMSE on available test TWS_t values: {score:.6f}")
print(f"Rows evaluated: {mask.sum():,} / {len(test):,}")

# RMSE on available test TWS_t values: 0.868617

RMSE on available test TWS_t values: 0.868912
Rows evaluated: 94,048 / 280,961


In [13]:
comparison = test.loc[mask, ["TWS_t", "test_pred"]].copy()
comparison["error"] = comparison["test_pred"] - comparison["TWS_t"]
comparison["abs_error"] = comparison["error"].abs()

print(comparison.head())
print(comparison["abs_error"].describe())

      TWS_t  test_pred     error  abs_error
0 -0.373966   0.415568  0.789533   0.789533
1 -0.530701   0.554694  1.085395   1.085395
2  0.531723   0.463599 -0.068124   0.068124
3  0.310179   0.209675 -0.100504   0.100504
4  0.110071   0.326394  0.216323   0.216323
count    94048.000000
mean         0.701911
std          0.512183
min          0.000006
25%          0.299067
50%          0.609232
75%          0.998526
max          3.834243
Name: abs_error, dtype: float64


In [14]:
missing = test["TWS_t"].isna()

test.loc[missing, "TWS_t"] = test.loc[missing, "test_pred"]

print(f"Filled {missing.sum():,} missing TWS_t values.")

Filled 186,913 missing TWS_t values.


In [15]:
train.isna().sum()

sample_id             0
time                  0
lat                   0
lon                   0
TWS_t                 0
SPEI_01_t             0
SPEI_03_t             0
SPEI_06_t             0
SPEI_12_t             0
SOIL_MOISTURE_t       0
month_sin             0
month_cos             0
target                0
SPEI_mean_t           0
SPEI_std_t            0
SPEI_median           0
SPEI_abs_mean         0
SPEI_abs_max          0
SPEI_energy           0
SPEI_cv               0
spei_sign_changes     0
spei_all_same_sign    0
SPEI01_div03          0
SPEI03_div06          0
SPEI06_div12          0
soil_abs              0
soil_cube             0
soil_inv              0
soil_x_SPEI_01_t      0
soil_div_SPEI_01_t    0
soil_x_SPEI_03_t      0
soil_div_SPEI_03_t    0
soil_x_SPEI_06_t      0
soil_div_SPEI_06_t    0
soil_x_SPEI_12_t      0
soil_div_SPEI_12_t    0
wetness_index         0
wetness_abs           0
severe_drought        0
moderate_drought      0
wet_condition         0
SPEI_01_t_sq    

In [16]:
test.isna().sum()

ID                    0
time                  0
lat                   0
lon                   0
TWS_t                 0
SPEI_01_t             0
SPEI_03_t             0
SPEI_06_t             0
SPEI_12_t             0
SOIL_MOISTURE_t       0
month_sin             0
month_cos             0
TWS_t_masked          0
SPEI_mean_t           0
SPEI_std_t            0
SPEI_median           0
SPEI_abs_mean         0
SPEI_abs_max          0
SPEI_energy           0
SPEI_cv               0
spei_sign_changes     0
spei_all_same_sign    0
SPEI01_div03          0
SPEI03_div06          0
SPEI06_div12          0
soil_abs              0
soil_cube             0
soil_inv              0
soil_x_SPEI_01_t      0
soil_div_SPEI_01_t    0
soil_x_SPEI_03_t      0
soil_div_SPEI_03_t    0
soil_x_SPEI_06_t      0
soil_div_SPEI_06_t    0
soil_x_SPEI_12_t      0
soil_div_SPEI_12_t    0
wetness_index         0
wetness_abs           0
severe_drought        0
moderate_drought      0
wet_condition         0
SPEI_01_t_sq    

In [19]:
# import numpy as np
# import pandas as pd


# def add_features(df):
#     df = df.copy()

#     # ----------------------------
#     # Time
#     # ----------------------------
#     df["time"] = pd.to_datetime(df["time"])

#     df["year"] = df["time"].dt.year
#     df["month"] = df["time"].dt.month
#     df["day"] = df["time"].dt.day
#     df["dayofyear"] = df["time"].dt.dayofyear
#     df["week"] = df["time"].dt.isocalendar().week.astype(int)
#     df["quarter"] = df["time"].dt.quarter
#     df["days_in_month"] = df["time"].dt.days_in_month
#     df["is_month_start"] = df["time"].dt.is_month_start.astype(np.int8)
#     df["is_month_end"] = df["time"].dt.is_month_end.astype(np.int8)

#     # ----------------------------
#     # Southern Hemisphere season
#     # ----------------------------
#     season_map = {
#         12: 0, 1: 0, 2: 0,
#          3: 1, 4: 1, 5: 1,
#          6: 2, 7: 2, 8: 2,
#          9: 3,10: 3,11: 3
#     }

#     df["season"] = df["month"].map(season_map)

#     # ----------------------------
#     # Cyclic time
#     # ----------------------------
#     df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
#     df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

#     df["day_sin"] = np.sin(2 * np.pi * df["dayofyear"] / 365.25)
#     df["day_cos"] = np.cos(2 * np.pi * df["dayofyear"] / 365.25)

#     # ----------------------------
#     # Latitude / Longitude
#     # ----------------------------
#     df["abs_lat"] = np.abs(df["lat"])
#     df["abs_lon"] = np.abs(df["lon"])

#     df["lat2"] = df["lat"] ** 2
#     df["lat3"] = df["lat"] ** 3

#     df["lon2"] = df["lon"] ** 2
#     df["lon3"] = df["lon"] ** 3

#     lat_rad = np.radians(df["lat"])
#     lon_rad = np.radians(df["lon"])

#     df["sin_lat"] = np.sin(lat_rad)
#     df["cos_lat"] = np.cos(lat_rad)

#     df["sin_lon"] = np.sin(lon_rad)
#     df["cos_lon"] = np.cos(lon_rad)

#     # ----------------------------
#     # Cartesian coordinates
#     # ----------------------------
#     df["x"] = np.cos(lat_rad) * np.cos(lon_rad)
#     df["y"] = np.cos(lat_rad) * np.sin(lon_rad)
#     df["z"] = np.sin(lat_rad)

#     # ----------------------------
#     # Geographic
#     # ----------------------------
#     df["pole_distance"] = 90 + df["lat"]

#     df["southern"] = (df["lat"] < 0).astype(np.int8)
#     df["western"] = (df["lon"] < 0).astype(np.int8)

#     df["lat_band"] = pd.cut(
#         df["lat"],
#         bins=[-90, -70, -60, -50, -40],
#         labels=False
#     )

#     df["lon_band"] = pd.cut(
#         df["lon"],
#         bins=np.arange(-180, 181, 5),
#         labels=False
#     )

#     # ----------------------------
#     # Interaction Features
#     # ----------------------------
#     df["lat_lon"] = df["lat"] * df["lon"]

#     df["lat_month"] = df["lat"] * df["month"]
#     df["lon_month"] = df["lon"] * df["month"]

#     df["lat_day"] = df["lat"] * df["dayofyear"]
#     df["lon_day"] = df["lon"] * df["dayofyear"]

#     df["season_lat"] = df["season"] * df["lat"]
#     df["season_lon"] = df["season"] * df["lon"]

#     df["month_lat"] = df["month"] * df["lat"]
#     df["month_lon"] = df["month"] * df["lon"]

#     df["day_lat"] = df["dayofyear"] * df["lat"]
#     df["day_lon"] = df["dayofyear"] * df["lon"]

#     df["lat_lon2"] = df["lat"] * (df["lon"] ** 2)
#     df["lat2_lon"] = (df["lat"] ** 2) * df["lon"]
#     df["lat2_lon2"] = (df["lat"] ** 2) * (df["lon"] ** 2)

#     return df

# import numpy as np
# import pandas as pd


# # Now i need the lag, shift ,expanding windows , moving average features etc alongside the TWS features 


# def add_hydrology_features(df):
#     df = df.copy()

#     # -----------------------------------------------------
#     # Current SPEI columns
#     # -----------------------------------------------------
#     spei_t = [
#         "SPEI_01_t",
#         "SPEI_03_t",
#         "SPEI_06_t",
#         "SPEI_12_t",
#     ]

#     # -----------------------------------------------------
#     # Mean / Std / Range
#     # -----------------------------------------------------
#     df["SPEI_mean_t"] = df[spei_t].mean(axis=1)
#     df["SPEI_std_t"] = df[spei_t].std(axis=1)

#     df["SPEI_min_t"] = df[spei_t].min(axis=1)
#     df["SPEI_max_t"] = df[spei_t].max(axis=1)

#     df["SPEI_range_t"] = (
#         df["SPEI_max_t"] -
#         df["SPEI_min_t"]
#     )

#     # -----------------------------------------------------
#     # Multi-scale drought differences
#     # -----------------------------------------------------
#     df["SPEI_short_long_t"] = (
#         df["SPEI_01_t"] -
#         df["SPEI_12_t"]
#     )

#     df["SPEI_01_03_t"] = (
#         df["SPEI_01_t"] -
#         df["SPEI_03_t"]
#     )

#     df["SPEI_03_06_t"] = (
#         df["SPEI_03_t"] -
#         df["SPEI_06_t"]
#     )

#     df["SPEI_06_12_t"] = (
#         df["SPEI_06_t"] -
#         df["SPEI_12_t"]
#     )

#     # -----------------------------------------------------
#     # TWS interactions
#     # -----------------------------------------------------
#     df["TWS_x_SOIL"] = (
#         df["TWS_t"] *
#         df["SOIL_MOISTURE_t"]
#     )

#     df["TWS_x_SPEI_mean"] = (
#         df["TWS_t"] *
#         df["SPEI_mean_t"]
#     )

#     df["TWS_div_SOIL"] = (
#         df["TWS_t"] /
#         (np.abs(df["SOIL_MOISTURE_t"]) + 1e-3)
#     )

#     # -----------------------------------------------------
#     # Soil × SPEI
#     # -----------------------------------------------------
#     df["SOIL_x_SPEI"] = (
#         df["SOIL_MOISTURE_t"] *
#         df["SPEI_mean_t"]
#     )

#     # -----------------------------------------------------
#     # Nonlinear transforms
#     # -----------------------------------------------------
#     df["TWS_sq"] = df["TWS_t"] ** 2

#     df["SOIL_sq"] = (
#         df["SOIL_MOISTURE_t"] ** 2
#     )

#     df["TWS_SOIL_diff"] = (
#         df["TWS_t"] -
#         df["SOIL_MOISTURE_t"]
#     )

#     df["TWS_SOIL_sum"] = (
#         df["TWS_t"] +
#         df["SOIL_MOISTURE_t"]
#     )

#     # -----------------------------------------------------
#     # Historical (safe) lag features
#     # -----------------------------------------------------
#     df = df.sort_values(
#         ["spatial_group", "time"]
#     ).reset_index(drop=True)

#     group = df.groupby("spatial_group")

#     df["TWS_t_lag1"] = group["TWS_t"].shift(1)
#     df["TWS_t_lag2"] = group["TWS_t"].shift(2)
#     df["TWS_t_lag3"] = group["TWS_t"].shift(3)

#     df["SOIL_lag1"] = group["SOIL_MOISTURE_t"].shift(1)

#     df["SPEI_01_lag1"] = group["SPEI_01_t"].shift(1)
#     df["SPEI_03_lag1"] = group["SPEI_03_t"].shift(1)
#     df["SPEI_06_lag1"] = group["SPEI_06_t"].shift(1)
#     df["SPEI_12_lag1"] = group["SPEI_12_t"].shift(1)

#     # -----------------------------------------------------
#     # Rolling statistics (history only)
#     # -----------------------------------------------------
#     df["TWS_roll_mean3"] = (
#         group["TWS_t"]
#         .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
#     )

#     df["TWS_roll_std3"] = (
#         group["TWS_t"]
#         .transform(lambda x: x.shift(1).rolling(3, min_periods=1).std())
#     )

#     df["SOIL_roll_mean3"] = (
#         group["SOIL_MOISTURE_t"]
#         .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
#     )

#     df["SPEI01_roll_mean3"] = (
#         group["SPEI_01_t"]
#         .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
#     )

#     return df


# def add_hydrology_physics_features(df):
#     df = df.copy()

#     eps = 1e-6

#     # =====================================================
#     # SPEI Scale Agreement
#     # =====================================================

#     df["SPEI_short_mean"] = (
#         df["SPEI_01_t"] +
#         df["SPEI_03_t"]
#     ) / 2

#     df["SPEI_long_mean"] = (
#         df["SPEI_06_t"] +
#         df["SPEI_12_t"]
#     ) / 2

#     df["SPEI_short_long_gap"] = (
#         df["SPEI_short_mean"] -
#         df["SPEI_long_mean"]
#     )

#     df["SPEI_scale_std"] = df[
#         [
#             "SPEI_01_t",
#             "SPEI_03_t",
#             "SPEI_06_t",
#             "SPEI_12_t",
#         ]
#     ].std(axis=1)

#     # =====================================================
#     # Drought Severity Indicators
#     # =====================================================

#     df["all_spei_negative"] = (
#         (
#             df[
#                 [
#                     "SPEI_01_t",
#                     "SPEI_03_t",
#                     "SPEI_06_t",
#                     "SPEI_12_t",
#                 ]
#             ] < 0
#         )
#         .all(axis=1)
#         .astype(np.int8)
#     )

#     df["all_spei_positive"] = (
#         (
#             df[
#                 [
#                     "SPEI_01_t",
#                     "SPEI_03_t",
#                     "SPEI_06_t",
#                     "SPEI_12_t",
#                 ]
#             ] > 0
#         )
#         .all(axis=1)
#         .astype(np.int8)
#     )

#     df["negative_spei_count"] = (
#         df[
#             [
#                 "SPEI_01_t",
#                 "SPEI_03_t",
#                 "SPEI_06_t",
#                 "SPEI_12_t",
#             ]
#         ] < 0
#     ).sum(axis=1)

#     # =====================================================
#     # Water Balance
#     # =====================================================

#     df["water_balance"] = (
#         df["TWS_t"] -
#         df["SOIL_MOISTURE_t"]
#     )

#     df["water_ratio"] = (
#         df["TWS_t"] /
#         (np.abs(df["SOIL_MOISTURE_t"]) + eps)
#     )

#     df["soil_fraction"] = (
#         df["SOIL_MOISTURE_t"] /
#         (np.abs(df["TWS_t"]) + eps)
#     )

#     # =====================================================
#     # Interaction Features
#     # =====================================================

#     df["TWS_x_SPEI01"] = (
#         df["TWS_t"] *
#         df["SPEI_01_t"]
#     )

#     df["TWS_x_SPEI03"] = (
#         df["TWS_t"] *
#         df["SPEI_03_t"]
#     )

#     df["TWS_x_SPEI06"] = (
#         df["TWS_t"] *
#         df["SPEI_06_t"]
#     )

#     df["TWS_x_SPEI12"] = (
#         df["TWS_t"] *
#         df["SPEI_12_t"]
#     )

#     df["SOIL_x_SPEI01"] = (
#         df["SOIL_MOISTURE_t"] *
#         df["SPEI_01_t"]
#     )

#     df["SOIL_x_SPEI12"] = (
#         df["SOIL_MOISTURE_t"] *
#         df["SPEI_12_t"]
#     )

#     # =====================================================
#     # Normalized Features
#     # =====================================================

#     df["TWS_norm"] = (
#         df["TWS_t"] /
#         (np.abs(df["SPEI_mean_t"]) + 1)
#     )

#     df["SOIL_norm"] = (
#         df["SOIL_MOISTURE_t"] /
#         (np.abs(df["SPEI_mean_t"]) + 1)
#     )

#     # =====================================================
#     # Hydrological Stress
#     # =====================================================

#     df["hydro_stress"] = (
#         np.abs(df["SPEI_mean_t"]) *
#         np.abs(df["water_balance"])
#     )

#     df["hydro_product"] = (
#         df["TWS_t"] *
#         df["SOIL_MOISTURE_t"] *
#         df["SPEI_mean_t"]
#     )

#     # =====================================================
#     # Relative SPEI Position
#     # =====================================================

#     df["SPEI01_minus_mean"] = (
#         df["SPEI_01_t"] -
#         df["SPEI_mean_t"]
#     )

#     df["SPEI12_minus_mean"] = (
#         df["SPEI_12_t"] -
#         df["SPEI_mean_t"]
#     )

#     # =====================================================
#     # Nonlinear
#     # =====================================================

#     df["sqrt_soil"] = np.sqrt(
#         np.abs(df["SOIL_MOISTURE_t"])
#     )

#     df["sqrt_tws"] = np.sqrt(
#         np.abs(df["TWS_t"])
#     )

#     df["log_soil"] = np.log1p(
#         np.abs(df["SOIL_MOISTURE_t"])
#     )

#     df["log_tws"] = np.log1p(
#         np.abs(df["TWS_t"])
#     )

#     return df


# train = add_hydrology_features(train)
# test = add_hydrology_features(test)


# train = add_hydrology_physics_features(train)
# test = add_hydrology_physics_features(test)



In [20]:
import numpy as np
import pandas as pd

def add_historical_features(df):
    df = df.copy()
    
    # Define SPEI columns used in the loop
    spei_t = [
        "SPEI_01_t",
        "SPEI_03_t",
        "SPEI_06_t",
        "SPEI_12_t",
    ]

    # -----------------------------------------------------
    # Historical features (SAFE)
    # Uses ONLY past observations within each spatial_group
    # -----------------------------------------------------
    df = df.sort_values(
        ["spatial_group", "time"]
    ).reset_index(drop=True)

    group = df.groupby("spatial_group")

    # =====================================================
    # LAG FEATURES
    # =====================================================

    for lag in [1, 2, 3, 6, 12]:

        df[f"TWS_lag{lag}"] = group["TWS_t"].shift(lag)
        df[f"SOIL_lag{lag}"] = group["SOIL_MOISTURE_t"].shift(lag)

        for c in spei_t:
            df[f"{c}_lag{lag}"] = group[c].shift(lag)

    # =====================================================
    # DIFFERENCE FEATURES
    # =====================================================

    for lag in [1, 2, 3]:

        df[f"TWS_diff{lag}"] = (
            df["TWS_t"] -
            df[f"TWS_lag{lag}"]
        )

        df[f"SOIL_diff{lag}"] = (
            df["SOIL_MOISTURE_t"] -
            df[f"SOIL_lag{lag}"]
        )

    # =====================================================
    # PERCENT CHANGE
    # =====================================================

    df["TWS_pct_change1"] = (
        group["TWS_t"]
        .pct_change()
    )

    df["SOIL_pct_change1"] = (
        group["SOIL_MOISTURE_t"]
        .pct_change()
    )

    # =====================================================
    # ROLLING MEAN
    # =====================================================

    for w in [3, 6, 12]:

        df[f"TWS_roll_mean{w}"] = (
            group["TWS_t"]
            .transform(
                lambda x:
                x.shift(1)
                 .rolling(w, min_periods=1)
                 .mean()
            )
        )

        df[f"SOIL_roll_mean{w}"] = (
            group["SOIL_MOISTURE_t"]
            .transform(
                lambda x:
                x.shift(1)
                 .rolling(w, min_periods=1)
                 .mean()
            )
        )

    # =====================================================
    # ROLLING STD
    # =====================================================

    for w in [3, 6, 12]:

        df[f"TWS_roll_std{w}"] = (
            group["TWS_t"]
            .transform(
                lambda x:
                x.shift(1)
                 .rolling(w, min_periods=2)
                 .std()
            )
        )

        df[f"SOIL_roll_std{w}"] = (
            group["SOIL_MOISTURE_t"]
            .transform(
                lambda x:
                x.shift(1)
                 .rolling(w, min_periods=2)
                 .std()
            )
        )

    # =====================================================
    # ROLLING MIN/MAX
    # =====================================================

    for w in [3, 6, 12]:

        df[f"TWS_roll_min{w}"] = (
            group["TWS_t"]
            .transform(
                lambda x:
                x.shift(1)
                 .rolling(w, min_periods=1)
                 .min()
            )
        )

        df[f"TWS_roll_max{w}"] = (
            group["TWS_t"]
            .transform(
                lambda x:
                x.shift(1)
                 .rolling(w, min_periods=1)
                 .max()
            )
        )

    # =====================================================
    # EXPANDING FEATURES
    # =====================================================

    df["TWS_exp_mean"] = (
        group["TWS_t"]
        .transform(
            lambda x:
            x.shift(1).expanding().mean()
        )
    )

    df["TWS_exp_std"] = (
        group["TWS_t"]
        .transform(
            lambda x:
            x.shift(1).expanding().std()
        )
    )

    df["SOIL_exp_mean"] = (
        group["SOIL_MOISTURE_t"]
        .transform(
            lambda x:
            x.shift(1).expanding().mean()
        )
    )

    # =====================================================
    # EXPONENTIAL MOVING AVERAGE
    # =====================================================

    for span in [3, 6, 12]:

        df[f"TWS_ema{span}"] = (
            group["TWS_t"]
            .transform(
                lambda x:
                x.shift(1)
                 .ewm(span=span, adjust=False)
                 .mean()
            )
        )

        df[f"SOIL_ema{span}"] = (
            group["SOIL_MOISTURE_t"]
            .transform(
                lambda x:
                x.shift(1)
                 .ewm(span=span, adjust=False)
                 .mean()
            )
        )

    # =====================================================
    # MOMENTUM
    # =====================================================

    df["TWS_momentum3"] = (
        df["TWS_lag1"] -
        df["TWS_lag3"]
    )

    df["TWS_momentum6"] = (
        df["TWS_lag1"] -
        df["TWS_lag6"]
    )

    df["SOIL_momentum3"] = (
        df["SOIL_lag1"] -
        df["SOIL_lag3"]
    )

    # =====================================================
    # DISTANCE FROM MOVING AVERAGE
    # =====================================================

    df["TWS_minus_roll3"] = (
        df["TWS_t"] -
        df["TWS_roll_mean3"]
    )

    df["TWS_minus_roll12"] = (
        df["TWS_t"] -
        df["TWS_roll_mean12"]
    )

    return df



train = add_historical_features(train)
test = add_historical_features(test)

KeyError: 'spatial_group'

In [31]:
# # Columns that should NOT be used as features
# DROP_COLS = [
#     "sample_id",
#     "Target",
#     "time",
#     "lat",
#     "lon",
#     "h3_cell",
#     "spatial_group",
#     "time_spatial_group","target",
# ]

# # All remaining columns become features
# FEATURES = [
#     col
#     for col in train.columns
#     if col not in DROP_COLS 
# ]

# print(f"Number of features: {len(FEATURES)}")
# print(FEATURES)


# Columns that should NOT be used as features
DROP_COLS = [
    "sample_id",
    "Target",
    "target",
    "time",
    # 'month_sin', 'month_cos'
    "lat",
    "lon",
    "h3_cell",
    "spatial_group",
    "time_spatial_group",
]

# All remaining columns become features
FEATURES = [
    col
    for col in train.columns
    if col not in DROP_COLS
    and col != "TWS_t_masked"   # exclude if present
]

print(f"Number of features: {len(FEATURES)}")
print(FEATURES)

Number of features: 121
['TWS_t', 'SPEI_01_t', 'SPEI_03_t', 'SPEI_06_t', 'SPEI_12_t', 'SOIL_MOISTURE_t', 'month_sin', 'month_cos', 'SPEI_mean_t', 'SPEI_std_t', 'SPEI_median', 'SPEI_abs_mean', 'SPEI_abs_max', 'SPEI_energy', 'SPEI_cv', 'spei_sign_changes', 'spei_all_same_sign', 'SPEI01_div03', 'SPEI03_div06', 'SPEI06_div12', 'soil_abs', 'soil_cube', 'soil_inv', 'soil_x_SPEI_01_t', 'soil_div_SPEI_01_t', 'soil_x_SPEI_03_t', 'soil_div_SPEI_03_t', 'soil_x_SPEI_06_t', 'soil_div_SPEI_06_t', 'soil_x_SPEI_12_t', 'soil_div_SPEI_12_t', 'wetness_index', 'wetness_abs', 'severe_drought', 'moderate_drought', 'wet_condition', 'SPEI_01_t_sq', 'SPEI_01_t_cube', 'SPEI_01_t_abs', 'SPEI_03_t_sq', 'SPEI_03_t_cube', 'SPEI_03_t_abs', 'SPEI_06_t_sq', 'SPEI_06_t_cube', 'SPEI_06_t_abs', 'SPEI_12_t_sq', 'SPEI_12_t_cube', 'SPEI_12_t_abs', 'SPEI_max_scale', 'SPEI_min_scale', 'SPEI_entropy', 'TWS_lag1', 'SOIL_lag1', 'SPEI_01_t_lag1', 'SPEI_03_t_lag1', 'SPEI_06_t_lag1', 'SPEI_12_t_lag1', 'TWS_lag2', 'SOIL_lag2', 'SPEI

    No. They do not need to be equal, and in your setup they are actually measuring different things.
    
    Let's interpret your numbers:
    
    Outer RMSE: 0.640945
    Global OOF RMSE: 0.501347
    
    A gap of ~0.14 RMSE is fairly large, so it's worth understanding why.
    
    What each metric measures
    
    Global OOF RMSE (0.501347)
    
    This is computed by combining predictions from all outer folds:
    
    Each prediction comes from a model trained on historical data.
    You then pool all outer validation predictions together and compute one RMSE.
    
    This estimates average performance across all outer folds.
    
    Outer RMSE (0.640945)
    
    This is the RMSE of one particular outer fold (likely the last one printed).
    
    If you only printed the last fold, you cannot compare it directly to the global OOF RMSE.
    
    Instead, you should collect every outer fold RMSE:
    
    outer_scores = []
    
    ...
    
    outer_rmse = rmse(y_outer_valid, pred)
    outer_scores.append(outer_rmse)
    
    print(outer_rmse)
    
    At the end:
    
    print("Outer RMSEs:", outer_scores)
    print("Mean Outer RMSE:", np.mean(outer_scores))
    print("Std:", np.std(outer_scores))
    
    The mean of the outer fold RMSEs is the quantity you should compare with the global OOF RMSE.
    
    If the mean outer RMSE is still much larger than the global OOF RMSE
    
    Suppose you get:
    
    Outer fold RMSEs
    0.47
    0.49
    0.52
    0.53
    0.64
    
    Mean = 0.53
    Global OOF = 0.50
    
    A small difference like this is normal because RMSE is nonlinear; averaging fold RMSEs is not identical to computing RMSE over all concatenated predictions.
    
    However, if you get something like:
    
    Mean Outer RMSE = 0.64
    Global OOF RMSE = 0.50
    
    then that's a sign to investigate. Common causes include:
    
    One outer fold is much harder than the others (distribution shift).
    The validation years differ substantially from the training years.
    The OOF predictions are not being written back correctly.
    Leakage in feature engineering or preprocessing.
    What you should focus on
    
    For model selection and estimating leaderboard performance, prioritize:
    
    Mean outer RMSE across all outer folds.
    Standard deviation of outer RMSE (stability).
    Global OOF RMSE as a secondary summary.
    
    For a forecasting competition like yours, the outer temporal CV is the most important estimate because it mimics predicting future time periods.
    
    So the first thing I'd recommend is printing all five outer RMSEs and their mean before drawing conclusions from the gap between 0.64 and 0.50.

Your problem is actually two problems combined:

Forecasting into the future (predict TWS_{t+1}).
Missing predictor values (TWS_t is missing for some test rows).

These should be handled separately.

What you have
Train
Month t
-------------------------------
TWS_t          ✓
SPEI_t         ✓
SOIL_t         ✓
Target=TWS_t+1 ✓

Some months are completely missing (e.g., no observations for certain year-months), but for the rows that exist, both TWS_t and TWS_{t+1} are known.

Test
Future Month t
-------------------------------
TWS_t          partly missing
SPEI_t         ✓
SOIL_t         ✓
Need TWS_t+1   ?

The future months have never appeared in training.

The forecasting problem

You want to learn

f(
    TWS_t,
    SPEI_t,
    SOIL_t,
    season,
    location
)
=
TWS_{t+1}

This is a standard one-step-ahead forecasting model.

The missing-value problem

Some test rows don't have TWS_t.

Before forecasting, estimate those missing values using only variables available at time t.

For example:

g(
    SPEI_t,
    SOIL_t,
    season,
    location
)
=
TWS_t

This is an imputation/reconstruction model.

The complete pipeline
                 TRAIN
──────────────────────────────────────

Model A
(SPEI, SOIL, season, location)
            │
            ▼
          TWS_t


Model B
(TWS_t, SPEI, SOIL, season, location)
                │
                ▼
             TWS_t+1



                 TEST
──────────────────────────────────────

Missing TWS_t?
        │
       Yes
        │
        ▼
Use Model A
        │
Predicted TWS_t
        │
        ▼
Complete feature vector
        │
        ▼
Use Model B
        │
        ▼
Predict TWS_t+1
What about the missing months in the training period?

For example:

2002-05
2002-06  ← missing
2002-07  ← missing
2002-08

This is not something you should fabricate by interpolation just to make the timeline continuous. Those months simply don't exist in the dataset. During training:

use the available observations,
create lag or rolling features only from previous available observations within each spatial group,
and let your temporal cross-validation reflect these gaps.
The biggest challenge

The hardest part is not the missing TWS_t.

The hardest part is generalizing to future years. The model must learn relationships such as:

drought indices (SPEI),
soil moisture,
seasonality,
geographic differences,

that continue to hold in unseen months and years.

That's why a rigorous temporal validation (e.g., walk-forward evaluation where validation months are strictly later than training months) is much more important than maximizing random CV performance.

A practical recommendation

For your dataset, I would build two LightGBM models:

Model A (reconstruction): Predict TWS_t from SPEI, SOIL_MOISTURE, seasonal features, and spatial features. Use it only to fill missing TWS_t values in the test set.
Model B (forecast): Predict TWS_{t+1} from TWS_t (observed or reconstructed), SPEI, SOIL_MOISTURE, seasonal features, spatial features, and carefully constructed historical lag/rolling features that use only past information within each spatial group.

This separation keeps the missing-data problem and the forecasting problem independent, making the pipeline easier to validate and less prone to leakage.

In [35]:
sub = pd.read_csv(
    "/kaggle/input/datasets/ramjasmaurya/full-zindi-dataset/SampleSubmission(1).csv"
)

# Align predictions by ID
sub = sub.drop(columns=["Target"], errors="ignore")

sub = sub.merge(
    test_df[["ID", "test_pred"]],
    on="ID",
    how="left",
)

sub = sub.rename(columns={"test_pred": "Target"})

# Check for missing predictions
assert sub["Target"].isna().sum() == 0, "Some IDs are missing predictions."

sub.to_csv("submission.csv", index=False)
sub

,ID,Target
0,20150901_-55.5_-68.5,-0.316363
1,20150901_-55.5_-67.5,-0.316849
2,20150901_-54.5_-71.5,0.328180
3,20150901_-54.5_-70.5,0.223070
4,20150901_-54.5_-69.5,0.053880
...,...,...
280956,20181201_83.5_-31.5,-0.027260
280957,20181201_83.5_-30.5,-0.018768
280958,20181201_83.5_-29.5,-0.041375
280959,20181201_83.5_-28.5,-0.060518


In [36]:
from IPython.display import HTML

HTML("""
<a href="submission.csv" download
   style="
       font-size:20px;
       padding:12px 20px;
       background:#4CAF50;
       color:white;
       text-decoration:none;
       border-radius:8px;">
📥 Download submission.csv
</a>
""")

from IPython.display import FileLink

FileLink("submission.csv")

/kaggle/working/submission.csv